In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import tensorflow as tf
from tensorflow import keras
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms,ToTensor,Lambda,Compose
from torchvision import datasets
import torch.optim as optim
import torchvision.models as models
from PIL import Image
import shutil
import random

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [3]:
train_dir = r"E:\Deep Learning\TransferLearning\data\cattle_split\train"
val_dir   = r"E:\Deep Learning\TransferLearning\data\cattle_split\val"
test_dir  = r"E:\Deep Learning\TransferLearning\data\cattle_split\test"
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=test_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=test_transform)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)



In [5]:
print("Classes:", train_dataset.classes)
print("Number of classes:", len(train_dataset.classes))

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


Classes: ['Amritmahal', 'Ayrshire', 'Bargur', 'Dangi', 'Deoni', 'Gir', 'Hallikar', 'Hariana', 'Himachali Pahari', 'Kangayam', 'Kankrej', 'Kenkatha', 'Khariar', 'Khillari', 'Konkan Kapila', 'Kosali', 'Krishna_Valley', 'Ladakhi', 'Lakhimi', 'Malnad_gidda', 'Mewati', 'Nari', 'Nimari', 'Ongole', 'Poda Thirupu', 'Pulikulam', 'Punganur', 'Purnea', 'Rathi', 'Red kandhari', 'Red_Sindhi', 'Sahiwal', 'Shweta Kapila', 'Tharparkar', 'Umblachery', 'Vechur', 'bachaur', 'badri', 'bhelai', 'dagri', 'gangatari', 'gaolao', 'ghumsari', 'kherigarh', 'malvi', 'motu', 'nagori', 'ponwar', 'siri', 'thutho']
Number of classes: 50
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


In [6]:
mobilenet = models.mobilenet_v2(pretrained=True)


e:\Deep Learning\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Deep Learning\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
for param in mobilenet.features.parameters():
    param.requires_grad = False


In [8]:
mobilenet.classifier

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

In [9]:
num_classes = len(train_dataset.classes)

mobilenet.classifier = nn.Sequential(
    nn.Linear(1280,640),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(640,280),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(280,num_classes)
)


In [10]:
print(mobilenet)


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.classifier.parameters(), lr=0.001)


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [13]:
mobilenet = mobilenet.to(device)



In [14]:
epochs = 10

for epoch in range(epochs):
    mobilenet.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = mobilenet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"[MobileNetV2] Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")


[MobileNetV2] Epoch [1/10] - Loss: 3.3715
[MobileNetV2] Epoch [2/10] - Loss: 2.5494
[MobileNetV2] Epoch [3/10] - Loss: 2.2695
[MobileNetV2] Epoch [4/10] - Loss: 2.1461
[MobileNetV2] Epoch [5/10] - Loss: 2.0521
[MobileNetV2] Epoch [6/10] - Loss: 1.9763
[MobileNetV2] Epoch [7/10] - Loss: 1.9536
[MobileNetV2] Epoch [8/10] - Loss: 1.8844
[MobileNetV2] Epoch [9/10] - Loss: 1.8231
[MobileNetV2] Epoch [10/10] - Loss: 1.8244


In [15]:
from sklearn.metrics import confusion_matrix, classification_report
from PIL import Image

In [16]:
def evaluate_model(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    return accuracy, all_labels, all_preds


In [17]:
val_accuracy, y_true, y_pred = evaluate_model(mobilenet, val_loader)

print(f"Validation Accuracy (MobileNetV2): {val_accuracy:.2f}%")


e:\Deep Learning\venv\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Validation Accuracy (MobileNetV2): 50.31%


In [18]:
mobilenet.eval()
infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [19]:
def predict_image(image_path, model, class_names):
    image = Image.open(image_path).convert("RGB") # type: ignore
    image = infer_transform(image)
    image = image.unsqueeze(0)  # type: ignore # add batch dimension
    image = image.to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probs, 1)

    return class_names[predicted.item()], confidence.item()


In [ ]:
image_path = r"E:\Deep Learning\TransferLearning\data\testImages\test_cattle9.webp"  # your new image
class_names = train_dataset.classes
for img in os.listdir(r"E:\Deep Learning\TransferLearning\data\testImages"):
    image_path = os.path.join(r"E:\Deep Learning\TransferLearning\data\testImages", img)

    breed, confidence = predict_image(image_path, mobilenet, class_names)

    print(f"Predicted Breed: {breed}")
    print(f"Confidence: {confidence*100:.2f}%")


Predicted Breed: Purnea
Confidence: 97.45%


In [23]:
import optuna


def objective(trial):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # -----------------------------
    # Hyperparameters to tune
    # -----------------------------
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])

    # -----------------------------
    # Data Loaders
    # -----------------------------
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # -----------------------------
    # Model
    # -----------------------------
    model = models.mobilenet_v2(pretrained=True)

    # Freeze backbone
    for param in model.features.parameters():
        param.requires_grad = False

    num_classes = len(train_dataset.classes)
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.last_channel, num_classes)
    )

    model = model.to(device)

    # -----------------------------
    # Loss
    # -----------------------------
    criterion = nn.CrossEntropyLoss()

    # -----------------------------
    # Optimizer
    # -----------------------------
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # -----------------------------
    # Training (SHORT for tuning)
    # -----------------------------
    model.train()
    for epoch in range(3):   # small epochs for tuning
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    accuracy = correct / total
    return accuracy


e:\Deep Learning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [24]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)


[I 2026-02-20 19:03:28,204] A new study created in memory with name: no-name-6ec2b670-e64f-496b-99ad-d345319a8112
e:\Deep Learning\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Deep Learning\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
e:\Deep Learning\venv\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
[I 2026-02-20 19:38:21,965] Trial 0 finished with value: 0.475 and parameters: 

In [25]:
print("Best Trial:")
print("Accuracy:", study.best_value)
print("Params:", study.best_params)


Best Trial:
Accuracy: 0.49296875
Params: {'lr': 0.0006817429423467297, 'weight_decay': 6.7981737394223816e-06, 'batch_size': 32, 'optimizer': 'Adam'}
